In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Terrain complexity via the ruggedness index

## Summary

Demonstrate and visualize the application of the ruggedness index RIX assessment on various artificial maps. 

## Results

### Key results

- Examples are sunny-weather examples: A planar map, an egg box map and a radial wave map. They show:
  - Maps and steep parts.
  - Radial ruggedness values.
  - Real-world applications are in `ionia/milet/elevation/trix/`.
- Critical parts:
  - Interpolation method of 2D maps to 1D rays. Artifacts introduced need to be identified.
    - Interpolation method `nearest` and level crossing sampling may behave weird in combination.
    - When sampling from outside the domain, throwing errors atm annoying. To be improved w/ diagnosis tools.
  - Floatiness of the slopes and the critical slope: The binary comparison is unstable if the slope along a ray is often close to the critical slope (check out the radial wave map for critical slopes 1e-2 and 1e-3).

### Details

1. Requirements:
   - A map given as a `xarray.DataArray` given in its own coordinate system with coordinates `("y", "x")`.
   - If the map is given in a particular `CRS`, then keep this one, too.
   - A location on the map encoded in a `LocationCCS` object (given as easting and northing). `zone` is for the computations.
1. Parameters:
   - `R_km`: Radius [km] of the disc of evaluation. TR6: 3.5km.
   - `dr_km`: Stepsize between points on each ray. `R_km` shall be a multiple of `dr_km`.
   - `levels`: Levels of elevation contour lines. TR6: >5m.
1. Free parameters:
   - `slope_critical`: Threshold [m/m] at which a slope is considered steep. This is no angle. TR6: 0.033.
1. Nomenclature:
   - `RayGeometry` is the ray starting from a given location in a given direction of length `R_km`.
   - `FieldSampler` samples from the map at given points.
     - Typically, these points are specified by `RayGeometry`.
   - `RayProfile` evaluates the map profile. Two possibilities to create atm:
     - `create_regular` to create it as equidistantly spaced grid.
     - `create_levelcrossing` to create it from a vertically defined grid.
   - `RayResult` determines rix and steep segments, provides a summary statistics and constructs segments for plotting from the given `RayProfile`.
   - Any slope used here is measured in [m/m] as in TR6, especially not [°].
1. Examples:
   - A planar map.
   - An eggbox map.
   - A radial wave map.

## Remarks

1. Plots:
   - `plot_profile_and_steep_segments`: The slopes and its mask of steep slopes represent left-labelled intervals, and are plotted for simplicity at their label, i.e. at the left bound. The artifact is of visual nature.

## Imports and Setup

In [ ]:
import numpy as np
import xarray
import pandas as pd
import geopandas as gpd
import shapely.geometry
import matplotlib.colors
import matplotlib.pyplot as plt

import ergaleiothiki
import phoibe.artificial_data.fields
from phoibe.geography.complexity.rix import RayGeometry, RayProfile, NaNPolicy
from phoibe.geography.complexity.rix import compute_regular_rix, analyse
from phoibe.geography.complexity.rix import RegularGridXYSampler
from phoibe.geography.complexity.rix import RayResult, RadialRixResult

In [ ]:
LAND_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "land_cmap", [(0.00, "#2e7d32"), (0.30, "#66bb6a"), (0.55, "#a1887f"), (0.75, "#9e9e9e"), (1.00, "#ffffff")]
)

## Helper functions

### Plotting

In [ ]:
def plot_geodata(da: xarray.DataArray, cmap=LAND_CMAP):
    """Plot the elevation map in colours."""
    figure = plt.figure()
    ax = figure.add_subplot(1, 1, 1)
    cnorm = matplotlib.colors.Normalize(vmin=da.min(), vmax=da.max())
    mesh = ax.pcolormesh(da["x"], da["y"], da, cmap=cmap, shading="auto", norm=cnorm)

    ax.grid(visible=True)

    cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7)
    cbar.set_label("Elevation [m]")
    plt.tight_layout()
    return figure, ax


def plot_profile_and_steep_segments(ray_profile, slope_critical):
    """Plot the elevation profile and its slopes along a ray, and marks steep parts."""
    r_m = ray_profile.r_m[:-1]
    z = ray_profile.z[:-1]
    dz = analyse.slopes(ray_profile)
    steep_mask = analyse.steep_mask(ray_profile, slope_critical=slope_critical)

    figure, axes = plt.subplots(1, 2, squeeze=True, figsize=(17, 7))
    axes[0].plot(r_m, z)
    axes[0].plot(r_m[steep_mask], z[steep_mask], c="r")
    axes[0].set_title(f"Profile along ray for {slope_critical=}")

    axes[1].plot(r_m, dz)
    axes[1].hlines(y=slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].hlines(y=-slope_critical, xmin=r_m[0], xmax=r_m[-1], color="r")
    axes[1].set_title(f"Slope along ray for {slope_critical=}")

    return figure, axes

### Computation and evaluation of intermediate and final results

In [ ]:
def evaluate_rix_for_ray(ray_profile: RayProfile, slope_critical: float):
    """Evaluate RIX and related numbers."""
    n_steep_segments = analyse.steep_mask(ray_profile, slope_critical).sum()
    n_segments = analyse.slopes(ray_profile).size
    rix = RayResult(profile=ray_profile, slope_critical=slope_critical).ruggedness
    message = (
        f"Number of steep segments {n_steep_segments} vs total number of segments {n_segments} vs. RIX {rix:2.4f}."
    )
    return message


def compute_levelcrossing_rix(location, sampler, n_angles, R_km, dr_km, slope_critical, levels):
    """For radial rays, determine their individual steepness fractions and steep segments."""
    angles = np.linspace(0, 360, n_angles, endpoint=False)
    ray_results = []

    for theta in angles:
        ray = RayGeometry.from_compass_regular(location=location, theta=theta, R_km=R_km, dr_km=dr_km)
        ray_profile = RayProfile.create_levelcrossing(
            ray=ray, sampler=sampler, nan_policy=NaNPolicy.TRUNCATE, levels=levels
        )
        ray_results.append(RayResult(profile=ray_profile, slope_critical=slope_critical))

    rix_results = RadialRixResult(rays=ray_results)
    return rix_results

## Idea

1. `RayGeometry` a grid on a ray starting from any `location` in direction `theta` (geographic convention) providing coordinates in internal (i.e. 1D) representation `r_m` and external (i.e. 2D) representation `xs` and `ys`, all measured in [m].
   - Grid may be regular specifying the radius `R_km` and stepsize `dr_km`.
   - Grid may be irregular providing the internal representation `r_m`.
2. `RegularGridXYSampler` allows to sample from any `xarray.DataArray` with coodinates `y` and `x` via the method `sample(xs, ys)`.
   - Sampling atm `linear` and `nearest` done by `xarray`.
3. `RayProfile` evaluates samples along some ray, that might be:
   1. `DiscreteRayProfile` that evaluates at equidistant points.
   2. `LevelCrossingProfile` that computes levelcrossings first, and then performs the evaluation.


Show the objects in action first.

In [ ]:
NX, NY = 21, 21
DX, DY = 10, 10
SLOPE_X, SLOPE_Y = 0.5, 0
LEVELS = [*np.arange(-21, 51, 2), 81]

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 0.2
DR_km = 0.01
THETA = 85

plane = phoibe.artificial_data.fields.make_planar_field(nx=NX, ny=NY, dx=DX, dy=DY, slope_x=SLOPE_X, slope_y=SLOPE_Y)
figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
display("Coordinate arrays:")
display(ray.xs, ray.ys)
elevation_sampler = RegularGridXYSampler(da=plane, method="linear")
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)
display("RayProfile (regular)")
display(ray_profile.z)

ray_profile = RayProfile.create_levelcrossing(
    ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR, levels=LEVELS
)
display("RayProfile (level crossing)")
display(ray_profile.r_m)
display(ray_profile.z)
RayResult(profile=ray_profile, slope_critical=0.1).describe()

## A planar map

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
SLOPE_X, SLOPE_Y = 0.5, 0
LEVELS = np.arange(-1000, 1000, 20)

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 1.5
DR_km = 0.01
THETA = 30

plane = phoibe.artificial_data.fields.make_planar_field(nx=NX, ny=NY, dx=DX, dy=DY, slope_x=SLOPE_X, slope_y=SLOPE_Y)
figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=plane, method="linear")
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.MASK)

ray_profile_cross = RayProfile.create_levelcrossing(
    ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.MASK, levels=LEVELS
)


slope_critical = 0.3
rayresult_regular = RayResult(profile=ray_profile, slope_critical=slope_critical)
rayresult_lcross = RayResult(profile=ray_profile_cross, slope_critical=slope_critical)
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
display(rayresult_regular.describe())
display(evaluate_rix_for_ray(ray_profile, slope_critical))
_ = plot_profile_and_steep_segments(ray_profile_cross, slope_critical)
display(rayresult_lcross.describe())
display(evaluate_rix_for_ray(ray_profile_cross, slope_critical))

slope_critical = 0.5
rayresult_regular = RayResult(profile=ray_profile, slope_critical=slope_critical)
rayresult_lcross = RayResult(profile=ray_profile_cross, slope_critical=slope_critical)
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
display(rayresult_regular.describe())
display(evaluate_rix_for_ray(ray_profile, slope_critical))
_ = plot_profile_and_steep_segments(ray_profile_cross, slope_critical)
display(rayresult_lcross.describe())
display(evaluate_rix_for_ray(ray_profile_cross, slope_critical))
display(analyse.steep_segment_indices(ray_profile, 0.1))
display(ray_profile.r_m, len(ray_profile.r_m))
display(analyse.steep_ray_segments(ray_profile, 0.1)[0].xy)
display(analyse.steep_segment_indices(ray_profile_cross, 0.1))
display(ray_profile_cross.r_m, len(ray_profile_cross.r_m))
display(analyse.steep_ray_segments(ray_profile_cross, 0.1)[0].xy)

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 0.25

kwargs_rix = {"sampler": elevation_sampler, "n_angles": N_ANGLES, "R_km": R_km, "dr_km": DR_km}
kwargs_profiler = {"levels": LEVELS}

result_regular = compute_regular_rix(LOCATION, slope_critical=SLOPE_CRITICAL, **kwargs_rix)

figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_regular.rix:.2f}")

result_regular.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_regular.plot_polar()


result_cross = compute_levelcrossing_rix(LOCATION, slope_critical=SLOPE_CRITICAL, levels=LEVELS, **kwargs_rix)

figure, ax = plot_geodata(da=plane)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_cross.rix:.2f}")

result_cross.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_cross.plot_polar()
result_cross.to_dataframe()

## Same planar map, different location

In [ ]:
LOCATION = ergaleiothiki.perdix.LocationCCS(easting=997, northing=965, zone=32)
R_km = 1
DR_km = 0.01
THETA = 90

ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
ray.r_m.shape
elevation_sampler = RegularGridXYSampler(da=plane, method="nearest")
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)
pd.DataFrame({"r": ray_profile.r_m, "z": ray_profile.z}).plot(x="r", y="z")

## An egg box map

In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ_X, FREQ_Y = 0.002, 0.002

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 2
DR_km = 0.01
THETA = 90

LEVELS = np.arange(-1, 1, 0.1)

eggbox = phoibe.artificial_data.fields.make_eggbox_field(nx=NX, ny=NY, dx=DX, dy=DY, freq_x=FREQ_X, freq_y=FREQ_Y)
figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=eggbox, method="linear")
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)

ray_profile_cross = RayProfile.create_levelcrossing(
    ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR, levels=LEVELS
)


slope_critical = 0.00075
rayresult_regular = RayResult(profile=ray_profile, slope_critical=slope_critical)
rayresult_lcross = RayResult(profile=ray_profile_cross, slope_critical=slope_critical)
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
display(rayresult_regular.describe())
display(evaluate_rix_for_ray(ray_profile, slope_critical))
_ = plot_profile_and_steep_segments(ray_profile_cross, slope_critical)
display(rayresult_lcross.describe())
display(evaluate_rix_for_ray(ray_profile_cross, slope_critical))

slope_critical = 0.009
rayresult_regular = RayResult(profile=ray_profile, slope_critical=slope_critical)
rayresult_lcross = RayResult(profile=ray_profile_cross, slope_critical=slope_critical)
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)
display(rayresult_regular.describe())
display(evaluate_rix_for_ray(ray_profile, slope_critical))
_ = plot_profile_and_steep_segments(ray_profile_cross, slope_critical)
display(rayresult_lcross.describe())
display(evaluate_rix_for_ray(ray_profile_cross, slope_critical))

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 0.00075
LEVELS = np.arange(-1, 1, 0.1)

kwargs_rix = {"sampler": elevation_sampler, "n_angles": N_ANGLES, "R_km": R_km, "dr_km": DR_km}
kwargs_profiler = {"levels": LEVELS}

result_regular = compute_regular_rix(LOCATION, slope_critical=SLOPE_CRITICAL, **kwargs_rix)

figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_regular.rix:.2f}")

result_regular.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_regular.plot_polar()


result_cross = compute_levelcrossing_rix(LOCATION, slope_critical=SLOPE_CRITICAL, levels=LEVELS, **kwargs_rix)

figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_cross.rix:.2f}")

result_cross.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_cross.plot_polar()

result_cross.to_dataframe()

In [ ]:
result_regular.describe()

In [ ]:
LOCATION = ergaleiothiki.perdix.LocationCCS(easting=75, northing=-215, zone=32)
R_km = 1.5

N_ANGLES = 72
SLOPE_CRITICAL = 0.0003
LEVELS = np.arange(-1, 1, 0.02)

kwargs_rix = {"sampler": elevation_sampler, "n_angles": N_ANGLES, "R_km": R_km, "dr_km": DR_km}
kwargs_profiler = {"levels": LEVELS}

result_regular = compute_regular_rix(LOCATION, slope_critical=SLOPE_CRITICAL, **kwargs_rix)

figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_regular.rix:.2f}")

result_regular.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_regular.plot_polar()


result_cross = compute_levelcrossing_rix(LOCATION, slope_critical=SLOPE_CRITICAL, levels=LEVELS, **kwargs_rix)

figure, ax = plot_geodata(da=eggbox)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_cross.rix:.2f}")

result_cross.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_cross.plot_polar()

In [ ]:
bad_ray = RayGeometry.from_compass_regular(location=LOCATION, theta=75, R_km=R_km, dr_km=DR_km)

bad_profile = RayProfile.create_levelcrossing(
    ray=bad_ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR, **kwargs_profiler
)
bad_profile.z

## A radial wave map


In [ ]:
NX, NY = 201, 201
DX, DY = 10, 10
FREQ = 4

LOCATION = ergaleiothiki.perdix.LocationCCS(easting=0, northing=0, zone=32)
R_km = 2
DR_km = 0.01
THETA = 90

radial_wave = phoibe.artificial_data.fields.make_radial_wave_field(nx=NX, ny=NY, dx=DX, dy=DY, freq=FREQ)
figure, ax = plot_geodata(da=radial_wave)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")

ray = RayGeometry.from_compass_regular(location=LOCATION, theta=THETA, R_km=R_km, dr_km=DR_km)
elevation_sampler = RegularGridXYSampler(da=radial_wave, method="linear")
ray_profile = RayProfile.create_regular(ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR)

ray_profile_cross = RayProfile.create_levelcrossing(
    ray=ray, sampler=elevation_sampler, nan_policy=NaNPolicy.ERROR, levels=LEVELS
)


slope_critical = 0.0075
rayresult_regular = RayResult(profile=ray_profile, slope_critical=slope_critical)
_ = plot_profile_and_steep_segments(ray_profile, slope_critical)

In [ ]:
N_ANGLES = 72
SLOPE_CRITICAL = 1e-2
LEVELS = np.arange(-1, 1, 0.1)

kwargs_rix = {"sampler": elevation_sampler, "n_angles": N_ANGLES, "R_km": R_km, "dr_km": DR_km}
kwargs_profiler = {"levels": LEVELS}

result_regular = compute_regular_rix(LOCATION, slope_critical=SLOPE_CRITICAL, **kwargs_rix)

figure, ax = plot_geodata(da=radial_wave)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_regular.rix:.2f}")

result_regular.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_regular.plot_polar()


result_cross = compute_levelcrossing_rix(LOCATION, slope_critical=SLOPE_CRITICAL, levels=LEVELS, **kwargs_rix)

figure, ax = plot_geodata(da=radial_wave)
ax.scatter(x=LOCATION.easting, y=LOCATION.northing, c="r")
ax.set_title(f"Map with rays and their steep segments, RIX={result_cross.rix:.2f}")

result_cross.steep_segments_geodataframe(crs=None).plot(ax=ax, color="r", linewidth=1, label="ray's steep part")
ax.legend()

ax_rix = result_cross.plot_polar()